## Start -> Chat -> End 
* here before user ask question to llm there is hitp which will confirm by human that this question can ask to llm 

In [12]:
import os
from langgraph.graph import StateGraph, START, END, add_messages
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langchain_groq.chat_models import ChatGroq
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

In [26]:
load_dotenv()

class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage], add_messages]


llm= ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    api_key= os.getenv("GROQ_API_KEY")
)


In [27]:
def chat_reply(state:ChatState):
    messages= [SystemMessage(content="You are question answering chat bot, and if you don't know about anything which is asked please tell that you don't know instead of telling irrelavent response"), *state["messages"]]

    decision= interrupt({
        "type":"approval",
        "reason":"Model is about to answer a user question.",
        "question": state['messages'][-1].content,
        "instruction":"Approval this question? yes/no"
    })
    # flow -> interrupt message will go to frontend and frontend also send a dict which has approved decision key

    if decision['approved']=="no":
        return {"messages":[AIMessage(content="Not approved.")]}
    
    else:
        result= llm.invoke(messages)
        return {"messages":[AIMessage(content=result.content)]}
    

In [28]:
checkpointer= InMemorySaver()

builder= StateGraph(ChatState)

builder.add_node('chat_point', chat_reply)


builder.add_edge(START, "chat_point")
builder.add_edge("chat_point", END)


graph= builder.compile(checkpointer)

CONFIG= {"configurable":{"thread_id":1}}

result= graph.invoke({"messages":[HumanMessage(content="How are you?")]}, config=CONFIG)

In [29]:
result
# here state has been paused

{'messages': [HumanMessage(content='How are you?', additional_kwargs={}, response_metadata={}, id='ddd7d509-8812-4cb1-b2b0-97139760f13d')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'How are you?', 'instruction': 'Approval this question? yes/no'}, id='eb6712dc65f605899cea1689ee86c19c')]}

In [30]:
message= result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'How are you?',
 'instruction': 'Approval this question? yes/no'}

In [31]:
user_status= input(f"Backend want {message['type']}.\nReason -> {message['reason']},\nQuestion: {message['question']}\nInstruction:{message['instruction']}")

In [32]:
# Resume the graph with the approval decision 

# this where backend 'decision' variable get frontend dict response where it was paused and this again invoke with command will unpause and give final response
final_result= graph.invoke(
    Command(resume={'approved': user_status}),
    config=CONFIG
)

In [33]:
final_result

{'messages': [HumanMessage(content='How are you?', additional_kwargs={}, response_metadata={}, id='ddd7d509-8812-4cb1-b2b0-97139760f13d'),
  AIMessage(content="I'm doing well, thank you for asking! I'm a large language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help with any questions you have. How can I assist you today?", additional_kwargs={}, response_metadata={}, id='822e37dd-758c-44bd-8e27-7b8888041c32', tool_calls=[], invalid_tool_calls=[])]}